# Source based Retriever

### 1. Wikipedia Retriever

Given a query, it uses Wikipedia API to fetch relevant/similar documents from Wikipedia.

How does it work:

The LangChain docs page doesn't mention the internal algorithm. Under the hood, WikipediaRetriever uses the wikipedia Python package, which calls the MediaWiki API. 

The chain is: LangChain WikipediaRetriever → wikipedia Python package → MediaWiki API (action=query, list=search) → CirrusSearch → Elasticsearch

So in short — it's not doing any vector/semantic search. It's classic BM25-based full-text search via Elasticsearch, with Wikipedia-specific ranking boosts layered on top.

In [ ]:
# !pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

In [3]:
from langchain_community.retrievers import WikipediaRetriever

In [4]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [5]:
# Define your query
query = "the rivalry between Magnus Carlsen and Hikaru Nakamura"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [ ]:
docs

[Document(metadata={'title': 'Sergey Karjakin', 'summary': "Sergey Alexandrovich Karjakin (born 12 January 1990) is a Russian chess grandmaster and politician. A chess prodigy, he previously held the record for the world's youngest ever grandmaster, having qualified for the title at the age of 12 years and 7 months. On 12 September 2024, he became a senator for Crimea in the Federation Council of Russia.\nKarjakin won the European U10 Chess Championship in 1999 and was the World U12 Chess Champion in 2001. He earned the International Master title at age 11 and was awarded his grandmaster title in 2003. He represented Ukraine at the Chess Olympiad in 2004, winning team and individual gold. He competed in two more Chess Olympiads for Ukraine and won the Corus chess tournament in 2009, before transferring to Russia. He has since represented Russia five times in the Chess Olympiad, winning individual gold in 2010. He also won team gold with Russia at the World Team Chess Championship in 20

In [17]:
docs[0].metadata

{'title': 'Sergey Karjakin',
 'summary': "Sergey Alexandrovich Karjakin (born 12 January 1990) is a Russian chess grandmaster and politician. A chess prodigy, he previously held the record for the world's youngest ever grandmaster, having qualified for the title at the age of 12 years and 7 months. On 12 September 2024, he became a senator for Crimea in the Federation Council of Russia.\nKarjakin won the European U10 Chess Championship in 1999 and was the World U12 Chess Champion in 2001. He earned the International Master title at age 11 and was awarded his grandmaster title in 2003. He represented Ukraine at the Chess Olympiad in 2004, winning team and individual gold. He competed in two more Chess Olympiads for Ukraine and won the Corus chess tournament in 2009, before transferring to Russia. He has since represented Russia five times in the Chess Olympiad, winning individual gold in 2010. He also won team gold with Russia at the World Team Chess Championship in 2013 and 2019.\nKarj

In [21]:
docs[0].page_content

'Sergey Alexandrovich Karjakin (born 12 January 1990) is a Russian chess grandmaster and politician. A chess prodigy, he previously held the record for the world\'s youngest ever grandmaster, having qualified for the title at the age of 12 years and 7 months. On 12 September 2024, he became a senator for Crimea in the Federation Council of Russia.\nKarjakin won the European U10 Chess Championship in 1999 and was the World U12 Chess Champion in 2001. He earned the International Master title at age 11 and was awarded his grandmaster title in 2003. He represented Ukraine at the Chess Olympiad in 2004, winning team and individual gold. He competed in two more Chess Olympiads for Ukraine and won the Corus chess tournament in 2009, before transferring to Russia. He has since represented Russia five times in the Chess Olympiad, winning individual gold in 2010. He also won team gold with Russia at the World Team Chess Championship in 2013 and 2019.\nKarjakin won the 2012 World Rapid Chess Cham

In [26]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
Sergey Alexandrovich Karjakin (born 12 January 1990) is a Russian chess grandmaster and politician. A chess prodigy, he previously held the record for the world's youngest ever grandmaster, having qualified for the title at the age of 12 years and 7 months. On 12 September 2024, he became a senator for Crimea in the Federation Council of Russia.
Karjakin won the European U10 Chess Championship in 1999 and was the World U12 Chess Champion in 2001. He earned the International Master title at age 11 and was awarded his grandmaster title in 2003. He represented Ukraine at the Chess Olympiad in 2004, winning team and individual gold. He competed in two more Chess Olympiads for Ukraine and won the Corus chess tournament in 2009, before transferring to Russia. He has since represented Russia five times in the Chess Olympiad, winning individual gold in 2010. He also won team gold with Russia at the World Team Chess Championship in 2013 and 2019.
Karjakin won the 2012

In [24]:
# Define your query
query1 = "movies directed by Christopher Nolan"

# Get relevant Wikipedia documents
docs1 = retriever.invoke(query1)

In [25]:
# Print retrieved content
for i, doc in enumerate(docs1):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
Sir Christopher Nolan is a British-American film director, screenwriter, and film producer. His feature directorial debut was the neo-noir crime thriller  Following (1998) which was made on a budget of $6,000. Two years later, he directed the psychological thriller Memento (2000) which starred Guy Pearce as a man suffering from anterograde amnesia (short-term memory loss) searching for his wife's killer. Similar to his debut feature it had a non-linear narrative structure, and was his breakthrough film. It was acclaimed by critics and was a surprise commercial success. For the film Nolan received his first nomination for the Directors Guild of America Award for Outstanding Directing – Feature Film, and for writing its screenplay he was nominated for the Academy Award for Best Original Screenplay. He next directed the mystery thriller remake Insomnia (2002) which starred Al Pacino, Robin Williams, and Hilary Swank. It was his first film for Warner Bros., and w

### 2. Vector Store Retriever

Given a query, it fetches relevant/similar documents from a vector store using semantic similarity and return them in Langchain Document format.

In [27]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

/Users/suryamgupta/Documents/Learning-LangChain/langvenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [28]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [29]:
# Step 2: Initialize embedding model
embedding_model = OpenAIEmbeddings()

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [30]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [31]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [32]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [33]:
# this is pretty much similar to vector store's similarity search method and also gives same resuls
results = vectorstore.similarity_search(query, k=2)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


#### But retrivers has more capabilities:
1. Since it is a runnable, it can be made into chains.
2. Vector store similarity search method uses a vanilla technique, later we will see how we can use advanced semantic similarity search algorithms like MMR, which is only possible with retrievers.

# Search Strategy based retriever

### 1. Maximal Marginal Relevance (MMR)

"how can we pick results that are not only relevant to the query but also different from each other?"

MMR is an information retrieval algorithm designed to reduce redundancy in retrieved results while maintaining high relevance to the query.

Basically, if we used for top 5 docs, and 3 of them are essentially the same and maybe just paraphrased versions of each other, that's kinda useless and we could have made do with just asking for 3 docs then.

In regular similarity search, we may get docs where: all docs are similar to each other, repeating same info, lacks diverse perspectives, etc.

MMR avoids that by: picking the most relevant doc first, then picking the next most relevant doc which is also least similar to the already selected docs, and so on...

This helps especially in RAG pipelines where: we want our context window to contain diverse but still relevant info. Also useful when docs are semantically overlapping.

In [35]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [36]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [37]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 1}  # k = top results, lambda_mult = relevance-diversity balance (between 0 and 1, 1 is same as regular similarity search/minimum diversity, and 0 for maximum diversity)
)

In [38]:
query = "What is langchain?"
results = retriever.invoke(query)

In [39]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
LangChain makes it easy to work with LLMs.

--- Result 3 ---
LangChain supports Chroma, FAISS, Pinecone, and more.


See how 1 and 2 are almost similar.

In [40]:
# with 0.5
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance (between 0 and 1, 1 is same as regular similarity search/minimum diversity, and 0 for maximum diversity)
)
query = "What is langchain?"
results = retriever.invoke(query)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
Embeddings are vector representations of text.

--- Result 3 ---
LangChain supports Chroma, FAISS, Pinecone, and more.


Nice!

### 2. Multiquery Retriever

Sometimes a single query might not capture all the ways information is phrased in your documents.

For example:

Query: "How can I stay healthy?"

Could mean:

- What should I eat?
- How often should I exercise?
- How can I manage stress?

A simple similarity search might miss documents that talk about those things but don't use the word "healthy."

How it works: 
1. Takes your original query
2. Uses an LLM (e.g., GPT-3.5) to generate multiple semantically different versions of that query
3. Performs retrieval for each sub-query
4. Combines and deduplicates the results

Example: 

"How can I stay healthy?"

LLM creates following subqueries and performs retrieval for all of them:

1. "What are the best foods to maintain good health?"
2. "How often should I exercise to stay fit?"
3. "What lifestyle habits improve mental and physical wellness?"
4. "How can I boost my immune system naturally?"
5. "What daily routines support long-term health?"

In [49]:
# from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI

In [44]:
# Relevant health & wellness documents - first 5 are health and wellness related, next 5 are general knowledge documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [45]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [46]:
# simple retriever
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [82]:
# multi query retriever
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOpenAI(model="gpt-3.5-turbo") # model which creates multiple sub queries
)

In [83]:
# Ambiguous query with words like energy and balance
query = "How to improve energy levels and maintain balance?"

In [84]:
# Retrieve results by both methods
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [85]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 5 ---
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************

--- Result 1 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 2 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 3 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 4 ---
Black holes bend spacetime and store immense gravitational energy.

--- Result

We see that in first result, the last one is not health related as it just got confused bcoz the query contained words like energy and balance, however, in multiquery, it ignores solar energy one and gives all 5 proper.

Update - ok after resampling multiple times idk what happened now multiquery doesnt even give 5 results asked, just gives 6 or 7 or 8 and that too solar energy one is now always in top 5, eh idk how good this is either.

### 3. ContextualCompressionRetriever

The Contextual Compression Retriever in LangChain is an advanced retriever that improves retrieval quality by compressing documents after retrieval content based on the user's query. keeping only the relevant content based on user's query.

Query: "What is photosynthesis?"

Retrieved Document (by a traditional retriever):

"The Grand Canyon is a famous natural site.
Photosynthesis is how plants convert light into energy.
Many tourists visit every year."

Problem:
- The retriever returns the entire paragraph
- Only one sentence is actually relevant to the query
- The rest is irrelevant noise that wastes context window and may confuse the LLM

What Contextual Compression Retriever does:

Returns only the relevant part, e.g.
"Photosynthesis is how plants convert light into energy."

How It Works

1. Base Retriever (e.g., FAISS, Chroma) retrieves N documents.
2. A compressor (usually an LLM) is applied to each document.
3. The compressor keeps only the parts relevant to the query.
4. Irrelevant content is discarded.

When to Use

- Your documents are long and contain mixed information
- You want to reduce context length for LLMs
- You need to improve answer accuracy in RAG pipelines

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
# from langchain_classic.retrievers import LLMChainExtractor
# from langchain.retrievers.document_compressors import LLMChainExtractor
# from langchain_community.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.document_compressors import LLMChainExtractor # (lots of versions and moving around of this class, this works for now in langchain 1.2.10)

In [96]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [97]:
# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding_model)

In [98]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [99]:
# Set up the compressor using an LLM
llm = ChatOpenAI(model="gpt-3.5-turbo")
compressor = LLMChainExtractor.from_llm(llm)

In [100]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [101]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [102]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.


Only 2 out of 4 results printed, which are both about photosynthesis, and also these 2 had multiple lines but only photosynthesis line is returned.